In [10]:
import pandas as pd
import numpy as np
import os

In [11]:


# 1. Define paths
raw_path = "../data/raw/weatherAUS.csv"
processed_dir = "../data/processed"
processed_path = os.path.join(processed_dir, "weather_processed.csv")

# 2. Create processed directory if it doesn't exist
os.makedirs(processed_dir, exist_ok=True)

# 3. Load the raw data
df_raw = pd.read_csv(raw_path)

# 4. Save a direct, untouched copy to the processed folder
df_raw.to_csv(processed_path, index=False)

print("--- Raw Data Copied to Processed Folder Successfully ---")
print(f"Original Raw Path: {raw_path}")
# This file is now our working copy that we will clean and update
print(f"Working Copy Created At: {processed_path}")

--- Raw Data Copied to Processed Folder Successfully ---
Original Raw Path: ../data/raw/weatherAUS.csv
Working Copy Created At: ../data/processed\weather_processed.csv


In [12]:
# 1. Load the working copy from the processed folder
df = pd.read_csv(processed_path)

# 2. Drop rows where 'RainTomorrow' or 'RainToday' is missing
df_cleaned = df.dropna(subset=['RainTomorrow', 'RainToday']).copy()

# 3. Map categorical targets to binary integers (0 and 1)
df_cleaned['RainTomorrow'] = df_cleaned['RainTomorrow'].map({'No': 0, 'Yes': 1})
df_cleaned['RainToday'] = df_cleaned['RainToday'].map({'No': 0, 'Yes': 1})

print("--- Missing Targets Removed ---")
print(f"Shape before dropping targets: {df.shape}")
print(f"Shape after dropping targets: {df_cleaned.shape}")

--- Missing Targets Removed ---
Shape before dropping targets: (145460, 23)
Shape after dropping targets: (140787, 23)


In [13]:
# Identify numerical features (excluding target columns)
numerical_features = df_cleaned.select_dtypes(include=['float64', 'int64']).columns
numerical_features = numerical_features.drop(['RainTomorrow', 'RainToday'])

# Impute missing numerical values using the median of each specific Location
for feature in numerical_features:
    df_cleaned[feature] = df_cleaned.groupby('Location')[feature].transform(lambda x: x.fillna(x.median()))

# Global fallback for features where an entire location has no records
for feature in numerical_features:
    if df_cleaned[feature].isnull().sum() > 0:
        df_cleaned[feature] = df_cleaned[feature].fillna(df_cleaned[feature].median())

print("--- Numerical Imputation Complete ---")

--- Numerical Imputation Complete ---


In [14]:
categorical_features = ['WindGustDir', 'WindDir9am', 'WindDir3pm']

# Impute missing categorical values using the mode of each specific Location
for feature in categorical_features:
    df_cleaned[feature] = df_cleaned.groupby('Location')[feature].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
    )

print("--- Categorical Imputation Complete ---")

--- Categorical Imputation Complete ---


In [15]:
# Overwrite the processed file with the fully cleaned and imputed dataset
df_cleaned.to_csv(processed_path, index=False)

print("--- Processed Data Successfully Updated and Saved! ---")
print(f"Final Cleaned File Location: {processed_path}")
print(f"Remaining Missing Values: {df_cleaned.isnull().sum().sum()}")

--- Processed Data Successfully Updated and Saved! ---
Final Cleaned File Location: ../data/processed\weather_processed.csv
Remaining Missing Values: 0


In [16]:
# Function to cap outliers using the IQR method
def cap_outliers_iqr(dataframe, features):
    df_capped = dataframe.copy()
    for col in features:
        Q1 = df_capped[col].quantile(0.25)
        Q3 = df_capped[col].quantile(0.75)
        IQR = Q3 - Q1

        # Define severe outlier boundaries
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Cap the values
        df_capped[col] = np.where(df_capped[col] < lower_bound, lower_bound, df_capped[col])
        df_capped[col] = np.where(df_capped[col] > upper_bound, upper_bound, df_capped[col])
    return df_capped

# Apply outlier capping to heavy numerical features
features_to_cap = ['Rainfall', 'Evaporation', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm']
df_cleaned = cap_outliers_iqr(df_cleaned, features_to_cap)

print("--- Outlier Capping via IQR Complete ---")
print("Extreme anomalies have been safely bounded to protect model stability.")

--- Outlier Capping via IQR Complete ---
Extreme anomalies have been safely bounded to protect model stability.


In [17]:
print("--- FINAL DATA SANITY CHECK ---")
print(f"1. Total Rows: {df_cleaned.shape[0]}")
print(f"2. Total Columns: {df_cleaned.shape[1]}")
print(f"3. Total Missing Values Left: {df_cleaned.isnull().sum().sum()}")
print("\n--- Remaining Missing Values Per Column ---")
print(df_cleaned.isnull().sum())
print("\n--- First 3 Rows of Cleaned Dataset ---")
df_cleaned.head(3)

--- FINAL DATA SANITY CHECK ---
1. Total Rows: 140787
2. Total Columns: 23
3. Total Missing Values Left: 0

--- Remaining Missing Values Per Column ---
Date             0
Location         0
MinTemp          0
MaxTemp          0
Rainfall         0
Evaporation      0
Sunshine         0
WindGustDir      0
WindGustSpeed    0
WindDir9am       0
WindDir3pm       0
WindSpeed9am     0
WindSpeed3pm     0
Humidity9am      0
Humidity3pm      0
Pressure9am      0
Pressure3pm      0
Cloud9am         0
Cloud3pm         0
Temp9am          0
Temp3pm          0
RainToday        0
RainTomorrow     0
dtype: int64

--- First 3 Rows of Cleaned Dataset ---


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,4.4,8.5,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,7.0,16.9,21.8,0,0
1,2008-12-02,Albury,7.4,25.1,0.0,4.4,8.5,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,8.0,7.0,17.2,24.3,0,0
2,2008-12-03,Albury,12.9,25.7,0.0,4.4,8.5,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,8.0,2.0,21.0,23.2,0,0
